###Usecase2: Telecom data - CDC, CDF, SCD

![](devices_cdc1.png)

**Prerequisites - One time activity (done by admin, based on the request)**

STEP 1: CREATE CONNECTION (FOREIGN CATALOG)

In [0]:
%sql
CREATE CONNECTION telecom_gcp_mysql_conn2
TYPE MYSQL
OPTIONS (
  host '35.223.80.16',
  port '3306',
  user 'inceptez',
  password 'Inceptez@123');

STEP 2: CREATE FOREIGN CATALOG

In [0]:
%sql
CREATE FOREIGN CATALOG telecom_federated_wd36_fc
USING CONNECTION telecom_gcp_mysql_conn2;

STEP 3: VERIFY SOURCE TABLE (LIVE QUERY)

In [0]:
%sql
SELECT * FROM telecom_federated_wd36_fc.telecom_devices.device_master;

STEP 4: CREATE LAYERS (BRONZE / SILVER / GOLD)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS telecom_bronze;
CREATE SCHEMA IF NOT EXISTS telecom_silver;
CREATE SCHEMA IF NOT EXISTS telecom_gold;

**Interview Questions wrt CDC, CDF, SCD1 & SCD2**

Change Data Feed (CDF)

Question: Why use Change Data Feed (CDF)?

Answer: It saves time and compute costs by only processing the rows that actually changed, instead of scanning the entire table.

Question: What does CDF output when a row is updated?

Answer: Two rows: the old version (pre-image) and the new version (post-image).

Slowly Changing Dimension Type 1 (SCD1)

Question: What is SCD Type 1?

Answer: It stores only the latest data. When a change happens, the old data is simply overwritten and lost.

Question: When should you use SCD Type 1?

Answer: When you only care about the current status and don't need to track historical changes (e.g., fixing a typo).

Slowly Changing Dimension Type 2 (SCD2)

Question: What is SCD Type 2?

Answer: It keeps the full history of changes. Instead of overwriting, it adds a new row for every update and uses tracking columns (like start/end dates) to show the timeline.

Question: How do you handle a "delete" in SCD Type 2?

Answer: You don't actually delete the row. You do a "soft delete" by marking the record as inactive and stamping an end date.

###Bronze Layer Load
STEP 5: HISTORICAL/INITIAL LOAD (FIRST TIME INGESTION)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS telecom_bronze.bronze_device_master2 (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, 
  owner_customer_id INT, status STRING, updated_at TIMESTAMP
);

--INSERT INTO telecom_bronze.bronze_device_master2
SELECT device_id, device_type, brand, model, os, owner_customer_id, status, updated_at 
FROM telecom_federated_wd36_fc.telecom_devices.device_master;

STEP 6: INCREMENTAL CDC LOAD (RUN MULTIPLE TIMES)

In [0]:
%sql
select * from telecom_bronze.bronze_device_master2

In [0]:
%sql
--Change Data Capture or Incremental Data ingestion feature using ???
INSERT INTO telecom_bronze.bronze_device_master2
SELECT 
    device_id, 
    device_type, 
    brand, 
    model, 
    os, 
    owner_customer_id, 
    status, 
    updated_at
FROM telecom_federated1.telecom_devices.device_master
WHERE updated_at > (    SELECT COALESCE(MAX(updated_at), '1900-01-01') 
    FROM telecom_federated_wd36_fc.telecom_devices.device_master);

In [0]:
%sql
select * from telecom_bronze.bronze_device_master2

###Silver Layer Load
STEP 7: CREATE SILVER TABLE WITH CDF ENABLED

CDF allows you to track and read incremental changes (inserts, updates, deletes) made to a Delta table — instead of reading the entire table every time.

In [0]:
%sql
select * from table_changes('telecom_silver.device_master_silver1', 0)

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS telecom_silver.silver_device_master2 (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, 
  owner_customer_id INT, status STRING, updated_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Step 1: Upsert (Insert + Update) from Bronze
MERGE INTO telecom_silver.silver_device_master2 AS target
USING (
  SELECT * FROM (
    -- Deduplicate Bronze: If a device came in 5 times today, only grab the latest one
    SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY updated_at DESC) as rank
    FROM telecom_bronze.bronze_device_master2
  ) WHERE rank = 1
) AS source
ON target.device_id = source.device_id
WHEN MATCHED AND target.updated_at != source.updated_at THEN 
  UPDATE SET *
WHEN NOT MATCHED THEN 
  INSERT *;

-- Step 2: Delete records no longer present in the live MySQL source
DELETE FROM telecom_silver.silver_device_master2
WHERE device_id NOT IN (
  SELECT device_id FROM telecom_federated_wd36_fc.telecom_devices.device_master
);

In [0]:
%sql
select * from table_changes('telecom_silver.silver_device_master2', 0)

### Gold Layer Load
**STEP 8: CREATE GOLD TABLE WITH SCD TYPE 1 & TYPE 2**

The Gold layer represents the final, business-ready state of your data, typically organized into dimensional models (facts and dimensions). To keep this layer updated efficiently, we utilize Delta Lake's Change Data Feed (CDF) from the Silver layer. 

CDF allows you to track and read incremental row-level changes (`insert`, `update_preimage`, `update_postimage`, `delete`) made to a Silver Delta table. Instead of scanning and recalculating the entire dataset every time a pipeline runs, you only process the exact changes, dramatically reducing compute costs and processing time for Slowly Changing Dimensions (SCD).

In [0]:
%sql
select * from telecom_gold.device_master_scd1

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS telecom_gold.device_master_scd1 (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, owner_customer_id INT, status STRING, updated_at TIMESTAMP, ingestion_time TIMESTAMP
);

MERGE INTO telecom_gold.device_master_scd1 AS target
USING (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY _commit_version DESC, _change_type DESC) as rank
    FROM table_changes('telecom_silver.silver_device_master', 0)
  ) WHERE rank = 1
) AS source
ON target.device_id = source.device_id
WHEN MATCHED AND source._change_type = 'delete' THEN 
  DELETE
WHEN MATCHED AND source._change_type IN ('insert', 'update_postimage') THEN 
  UPDATE SET 
    target.device_type = source.device_type,
    target.brand = source.brand,
    target.model = source.model,
    target.os = source.os,
    target.owner_customer_id = source.owner_customer_id,
    target.status = source.status,
    target.updated_at = source.updated_at,
    target.ingestion_time = current_timestamp()
WHEN NOT MATCHED AND source._change_type IN ('insert', 'update_postimage') THEN 
  INSERT (device_id, device_type, brand, model, os, owner_customer_id, status, updated_at, ingestion_time)
  VALUES (source.device_id, source.device_type, source.brand, source.model, source.os, source.owner_customer_id, source.status, source.updated_at, current_timestamp());

In [0]:
%sql
select * from telecom_gold.device_master_scd1

In [0]:
%sql
select * from telecom_gold.device_master_scd2

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS telecom_gold.device_master_scd2 (
  device_id INT, device_type STRING, brand STRING, model STRING, os STRING, owner_customer_id INT, status STRING,
  is_current BOOLEAN, start_date TIMESTAMP, end_date TIMESTAMP
);

WITH silver_changes AS (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY _commit_version DESC, _change_type DESC) as rank
    FROM table_changes('telecom_silver.silver_device_master', 0)
    WHERE _change_type IN ('insert', 'update_postimage', 'delete')
  ) WHERE rank = 1
),
scd2_merge_data AS (
  SELECT device_id as merge_key, * FROM silver_changes
  UNION ALL
  SELECT NULL as merge_key, * FROM silver_changes WHERE _change_type = 'update_postimage'
)

MERGE INTO telecom_gold.device_master_scd2 AS target
USING scd2_merge_data AS source
ON target.device_id = source.merge_key

-- Close out old records
WHEN MATCHED AND target.is_current = true AND source._change_type IN ('update_postimage', 'delete') THEN
  UPDATE SET target.is_current = false, target.end_date = source.updated_at

-- Insert new records
WHEN NOT MATCHED AND source._change_type IN ('insert', 'update_postimage') THEN
  INSERT (device_id, device_type, brand, model, os, owner_customer_id, status, is_current, start_date, end_date)
  VALUES (source.device_id, source.device_type, source.brand, source.model, source.os, source.owner_customer_id, source.status, true, source.updated_at, '9999-12-31T23:59:59.000Z');

In [0]:
%sql
select * from telecom_gold.device_master_scd2
